# ⚖️ 台灣司法官律師國考 評分與申論推導模型微調 (Fine-Tuning Law Exam Evaluator)
支援在 **Google Colab (免費 T4 GPU 或 CPU)** 上極速微調開源模型（如 Qwen2.5-7B 或 Taiwan-LLM）。
訓練資料直接從您的 Google Drive (`/content/drive/MyDrive/LawExamData/training_data`) 讀取 **100% 真實核實裁判、憲法法庭判決與法理規準語料**。

In [ ]:
# 1. 掛載 Google 雲端硬碟 (直接讀取既有數據，無需重複上傳)
import os
from google.colab import drive
drive.mount('/content/drive')

#@title 選擇微調資料集規模 (Dataset Selection)
#@markdown 支援「100萬筆全真統一大庫 (推薦深度微調)」或「5,064筆純核實核心庫 (推薦快速試跑)」：
DATASET_MODE = "100萬筆全真統一大庫 (1M+ Unified SFT)" #@param ["100萬筆全真統一大庫 (1M+ Unified SFT)", "5,064筆純核實核心庫 (Verified SFT)"]

file_name = "law_exam_master_sft_unified.jsonl" if "100萬" in DATASET_MODE else "law_exam_verified_sft.jsonl"

possible_paths = [
    f'/content/drive/MyDrive/LawExamData/training_data/{file_name}',
    f'/content/drive/我的雲端硬碟/LawExamData/training_data/{file_name}',
    f'/content/drive/MyDrive/LawExamData/training_data/law_exam_1m_judicial_reasoning_sft.jsonl',
    f'/content/drive/我的雲端硬碟/LawExamData/training_data/law_exam_1m_judicial_reasoning_sft.jsonl'
]
DATA_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break
if not DATA_PATH:
    DATA_PATH = possible_paths[0]

print('✅ 訓練資料集已就緒：', DATA_PATH)


In [ ]:
# 2. 安裝極速微調套件 (包含 unsloth_zoo 與最新依賴)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets triton

import importlib
importlib.invalidate_caches()
print('✅ Unsloth 與 unsloth_zoo 安裝完成！若初次安裝建議重啟工作階段 (Runtime -> Restart session)')


In [ ]:
# 3. 載入基底開源模型 (Qwen2.5-7B-Instruct 4-bit)
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
import torch

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# 配置高效能 LoRA 轉接器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


In [ ]:
# 4. 載入資料並格式化為 ChatML 指令提示詞
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset('json', data_files=DATA_PATH, split='train')

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = (
            f"<|im_start|>system\n你是一位具備多年命題閱卷經驗之台灣司法官律師考試閱卷委員與法學權威。<|im_end|>\n"
            f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        texts.append(text)
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"成功載入並格式化 {len(dataset)} 筆真實司律微調樣本！")

In [ ]:
# 5. 啟動 SFT 訓練迴圈
from unsloth import is_bfloat16_supported

#@title 設定訓練步數 (Training Steps)
#@markdown 針對百萬資料集，建議設定 120 ~ 1,000 steps（每 100 steps 約 5~10 分鐘）：
max_training_steps = 200 #@param {type:"integer"}

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = max_training_steps,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print('微調訓練完成！統計資訊：', trainer_stats)


In [ ]:
# 6. 將微調完成的 LoRA 權重儲存回 Google Drive
SAVE_DIR = '/content/drive/MyDrive/LawExamData/model_weights/qwen_law_evaluator_lora'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('🎉 司律評分模型 LoRA 權重已永久儲存至 Google Drive：', SAVE_DIR)